# 07 — Edge Inference and Benchmarking

## Goal

Run the exported LiteRT/TFLite model locally, measure inference latency, and inspect its input/output quantization parameters. The same inference pattern can later be used on a Raspberry Pi.

This notebook runs on your development machine first. It does not imply that the model has been deployed to a physical satellite or flight-qualified hardware.


In [ ]:
import os
import time
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import cifar10

MODEL_PATH = "../artifacts/cifar10_int8.tflite"
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "Quantized model not found. Run Notebook 06 first."
    )

(_, _), (x_test_raw, y_test_raw) = cifar10.load_data()
x_test = x_test_raw.astype(np.float32) / 255.0
y_test = y_test_raw.squeeze()

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

print("Input details:", input_details)
print("Output details:", output_details)


## 1. Prepare inputs correctly

For a float model, pass float32 inputs directly. For an INT8 model, convert normalized inputs using the tensor's quantization scale and zero point. Quantization metadata should be read from the model, not hard-coded.


In [ ]:
def prepare_input(image, details):
    dtype = details["dtype"]
    if dtype in (np.int8, np.uint8):
        scale, zero_point = details["quantization"]
        if scale == 0:
            raise ValueError("Invalid input quantization scale.")
        quantized = np.round(image / scale + zero_point)
        limits = np.iinfo(dtype)
        return np.clip(quantized, limits.min, limits.max).astype(dtype)
    return image.astype(dtype)

def decode_output(output, details):
    if details["dtype"] in (np.int8, np.uint8):
        scale, zero_point = details["quantization"]
        return (output.astype(np.float32) - zero_point) * scale
    return output.astype(np.float32)

sample = x_test[0:1]
prepared = prepare_input(sample, input_details)
print("Original input shape/dtype:", sample.shape, sample.dtype)
print("Model input shape/dtype:", prepared.shape, prepared.dtype)


## 2. Run one inference

The model expects a batch dimension, so a single image has shape `(1, 32, 32, 3)`.


In [ ]:
interpreter.set_tensor(input_details["index"], prepared)
interpreter.invoke()

raw_output = interpreter.get_tensor(output_details["index"])
scores = decode_output(raw_output, output_details)
predicted_class = int(np.argmax(scores[0]))

print("Predicted class:", CLASS_NAMES[predicted_class])
print("True class:", CLASS_NAMES[int(y_test[0])])
print("Top score:", float(np.max(scores[0])))


## 3. Measure latency and approximate accuracy

This is a simple single-image benchmark on the current machine. Latency depends on hardware, runtime, thermal conditions, and threading. The result is not a Raspberry Pi benchmark unless you run this notebook on the Pi.


In [ ]:
N_BENCHMARK = 200
n_eval = 500
correct = 0
latencies_ms = []

for i in range(n_eval):
    image = x_test[i:i+1]
    model_input = prepare_input(image, input_details)

    start = time.perf_counter()
    interpreter.set_tensor(input_details["index"], model_input)
    interpreter.invoke()
    elapsed_ms = (time.perf_counter() - start) * 1000
    latencies_ms.append(elapsed_ms)

    output = interpreter.get_tensor(output_details["index"])
    scores = decode_output(output, output_details)
    pred = int(np.argmax(scores[0]))
    correct += int(pred == int(y_test[i]))

latencies = np.array(latencies_ms)
print(f"Accuracy on first {n_eval} test images: {correct / n_eval:.3f}")
print(f"Mean latency: {latencies.mean():.2f} ms")
print(f"Median latency: {np.median(latencies):.2f} ms")
print(f"95th percentile latency: {np.percentile(latencies, 95):.2f} ms")


## 4. Edge deployment checklist

Before moving the model to a Raspberry Pi:

- Copy the `.tflite` file to the Pi.
- Install a compatible LiteRT/TFLite runtime for the Pi's OS, CPU architecture, and Python version.
- Confirm the expected input shape, data type, scale, and zero point.
- Reuse exactly the same resizing and normalization used during training.
- Measure latency and memory use on the target device.
- Record model size, accuracy, mean latency, and 95th-percentile latency.

## Final reflection

1. Why might a model be slower on a laptop CPU than on its GPU, or vice versa?
2. Why should we benchmark on the actual target device?
3. What would need to change to adapt this pipeline from CIFAR-10 to EuroSAT?
4. What additional tests would be needed before using a model in a real mission?

**Reminder:** this is an educational edge-ML prototype, not flight-qualified software.
